In [1]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [2]:
sys.path.append("../../experiments/parametrization_experiments/")

In [3]:
import parametrization_experiment_helper

In [4]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [5]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)
from scipy.spatial import ConvexHull, convex_hull_plot_2d


In [6]:
kappa_path = None
import json

In [7]:
import matplotlib.cm as cm


In [8]:
data_collection = []
labels = []
for pattern in parametrization_experiment_helper.Pattern_data:
    experiment_file = pattern['experiment_file']
    stiffness_path = pattern['stiffness_path']
    pattern_name = pattern['name']
    num_pattern_params = pattern['num_pattern_params']
    param_index = pattern['param_index']
    default_param = pattern['default_param']
    param_range = pattern['param_range']
    param_normalization_factor = pattern['param_normalization_factor']
    fusing_curve_polyline = pattern['fusing_curve_polyline_function']
    label = pattern['label']

    with open(experiment_file, 'r') as fp:
        data = json.load(fp)

    df = pd.DataFrame(data['data'])
    valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])


    bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, pattern_name, valid_tags, plot_data = False)

    max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
    min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
    max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
    min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)
    x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, pattern_name, valid_tags)
    min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
    max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
    angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, pattern_name, valid_tags)
    import copy
    data_collection.append([x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness])
    labels.append(label)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yren/Develop/Inflatables/python/periodic_patches/experiments/parallelized_experiments/output/dash_line/2024_01_21_21_08//experiment_result.json'

In [9]:
labels

[]

In [44]:
data_collection = [data_collection[2], data_collection[0], data_collection[3], data_collection[1]]
labels = [labels[2], labels[0], labels[3], labels[1]]

In [53]:
labels

In [54]:
single_colors = [cm.tab10(2), cm.tab10(0), cm.tab10(6), cm.tab10(1)]
colors = ['Blues', 'Oranges', 'Greens', 'PuRd',  'Greys', 'Purples', 
                      'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                      'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']
colors = [colors[2], colors[0], colors[3], colors[1]]


In [55]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (6, 6))

# labels = ['Dash Line', 'Double Dash Line', 'Cosine Curve', 'Ellipse holes angle', 'Ellipse holes width', 'Ellipse fused', 'Random Voronoi', 'Ellipse holes angle width']
for i in range(len(data_collection)):
    curr_data = data_collection[i]
    
    [x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness] = curr_data

    # plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
    # plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

    # plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)

    print(max(y_scale_factors))
    points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
    hull = ConvexHull(points)

    # for simplex in hull.simplices:
    #     plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

    ax.title.set_text("Scale factors")
    plt.xlabel("x scale factors")
    plt.ylabel("y scale factors")
    
    vmin = np.min(min_bending_stiffness)
    vmax = np.max(min_bending_stiffness)
    vmin -= (vmax - vmin) / 3

    scatter = plt.scatter(x_scale_factors, y_scale_factors, label = labels[i], s = 100, alpha = 0.8, c = min_bending_stiffness, cmap = colors[i], edgecolors='white', vmin = vmin, vmax = vmax, linewidths = 0.5)

# # Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# # now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
# ax.set_xlim(lims)
# ax.set_ylim(lims)
ax.legend()
leg = ax.get_legend()
for i in range(len(data_collection)):
    leg.legend_handles[i].set_color(cm.get_cmap(colors[i])(1.0))

fig.tight_layout()
plt.savefig('all_pattern_scale_factor_values.png', dpi = 300)
plt.savefig('all_pattern_scale_factor_values.svg', dpi = 300)

In [56]:
cm.tab10(0)

In [57]:
import numpy.linalg as la


fig, ax = plt.subplots(figsize = (6, 6))

# labels = ['Dash Line', 'Double Dash Line', 'Cosine Curve', 'Ellipse holes angle', 'Ellipse holes width', 'Ellipse fused', 'Random Voronoi', 'Ellipse holes angle width']
for i in range(len(data_collection)):
    curr_data = data_collection[i]
    
    [x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness] = curr_data

    print(max(y_scale_factors))
    points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
    hull = ConvexHull(points)

    # for simplex in hull.simplices:
    #     plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

    ax.title.set_text("Bending Stiffness")
    plt.xlabel("max. bending stiffness")
    plt.ylabel("min. bending stiffness")

    # plt.scatter(max_bending_stiffness, min_bending_stiffness, label = labels[i], s = 100, alpha = 0.4, color = single_colors[i])

    vmin = np.min(x_scale_factors)
    vmax = np.max(x_scale_factors)
    vmin -= (vmax - vmin) / 3
    scatter = plt.scatter(max_bending_stiffness, min_bending_stiffness, label = labels[i], s = 100, alpha = 0.8, c = x_scale_factors, cmap = colors[i], edgecolors='white', vmin = vmin, vmax = vmax, linewidths = 0.5)

# ax.set_aspect('equal')

ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(loc='upper left')
fig.tight_layout()
plt.savefig('all_pattern_bending_stiffness_values.png', dpi = 300)
plt.savefig('all_pattern_bending_stiffness_values.svg', dpi = 300)


In [58]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (6, 6))

single_colors = [cm.tab10(0), cm.tab10(1), cm.tab10(2), cm.tab10(6)]

# labels = ['Dash Line', 'Double Dash Line', 'Cosine Curve', 'Ellipse holes angle', 'Ellipse holes width', 'Ellipse fused', 'Random Voronoi', 'Ellipse holes angle width']
for i in range(len(data_collection)):
    curr_data = data_collection[i]
    
    [x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness] = curr_data

    print(max(y_scale_factors))
    points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
    hull = ConvexHull(points)

    # for simplex in hull.simplices:
    #     plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

    ax.title.set_text("Stretching stiffness")
    plt.xlabel("max. stretching stiffness")
    plt.ylabel("min. stretching stiffness")

    vmin = np.min(x_scale_factors)
    vmax = np.max(x_scale_factors)
    vmin -= (vmax - vmin) / 3
    scatter = plt.scatter(max_stretching_stiffness, min_stretching_stiffness, label = labels[i], s = 100, alpha = 0.8, c = x_scale_factors, cmap = colors[i], edgecolors='white', vmin = vmin, vmax = vmax, linewidths = 0.5)


# ax.set_aspect('equal')

ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(loc='lower right')
fig.tight_layout()
plt.savefig('all_pattern_stretching_stiffness_values.png', dpi = 300)
plt.savefig('all_pattern_stretching_stiffness_values.svg', dpi = 300)

In [59]:
import sklearn

from sklearn.manifold import TSNE

# Assuming data_collection is a list of 2D arrays, where each 2D array contains the data points for a pattern family
analyze_data_collection = [np.array(data).T for data in data_collection]

data_length = [len(np.array(data).T) for data in data_collection]
# Concatenate all the data points into a single 2D array
analyze_data_collection = np.concatenate(analyze_data_collection)

analyze_data_collection.shape

# Create a t-SNE object
tsne = TSNE(n_components=2)

# Fit the t-SNE model and transform the data
data_2d = tsne.fit_transform(analyze_data_collection)

# data_2d now contains the 2D projection of your data

data_length

result_data_collection = []
total_length = 0
for length in data_length:
    result_data_collection.append(data_2d[total_length:total_length + length])
    total_length += length

import numpy.linalg as la

fig, ax = plt.subplots(figsize = (6, 6))

single_colors = [cm.tab10(0), cm.tab10(1), cm.tab10(2), cm.tab10(6)]

for i in range(len(result_data_collection)):
    curr_data = result_data_collection[i]
    
    ax.title.set_text("t-SNE")


    plt.scatter(curr_data[:, 0], curr_data[:, 1], label = labels[i], s = 200, alpha = 0.4, color = single_colors[i])

# ax.set_aspect('equal')

ax.legend()
fig.tight_layout()
plt.savefig('t-SNE.png', dpi = 300)
plt.savefig('t-SNE.svg', dpi = 300)